In [3]:
import pandas as pd
import joblib
from datetime import datetime

X_test = pd.read_csv("../data/X_test.csv")
y_test = pd.read_csv("../data/y_test.csv").squeeze()
best_xgb = joblib.load("../models/xgboost_tuned_final.pkl")

def risk_tier(prob):
    if prob < 0.10:
        return "Low"
    elif prob < 0.50:
        return "Medium"
    else:
        return "High"

def recommend_action_with_rule(tier, is_high_value, tenure_segment):
    if tier == "Low":
        return "No action needed", "RULE_LOW_NO_ACTION"
    if tier == "High":
        if is_high_value and tenure_segment == "Loyal":
            return "Proactive support outreach", "RULE_HIGH_VALUE_LOYAL_OUTREACH"
        elif is_high_value:
            return "Personalized discount", "RULE_HIGH_VALUE_DISCOUNT"
        else:
            return "Plan downgrade offer", "RULE_LOW_VALUE_DOWNGRADE"
    if tier == "Medium":
        if is_high_value:
            return "Loyalty reward", "RULE_MEDIUM_HIGH_VALUE_REWARD"
        else:
            return "Proactive support outreach", "RULE_MEDIUM_LOW_VALUE_OUTREACH"

In [4]:
churn_probs = best_xgb.predict_proba(X_test)[:, 1]

audit_records = []
already_actioned_customers = set()

for idx in range(len(X_test)):
    customer_id = X_test.index[idx]

    if customer_id in already_actioned_customers:
        continue

    row = X_test.iloc[idx]
    prob = float(churn_probs[idx])
    tier = risk_tier(prob)
    is_high_value = bool(row["is_high_value_customer"])

    tenure_months = row["customer_tenure_months"]
    if tenure_months <= 6:
        tenure_segment = "New"
    elif tenure_months <= 24:
        tenure_segment = "Established"
    else:
        tenure_segment = "Loyal"

    action, rule_fired = recommend_action_with_rule(tier, is_high_value, tenure_segment)

    if action != "No action needed":
        already_actioned_customers.add(customer_id)

    audit_records.append({
        "customer_id": customer_id,
        "timestamp": datetime.now().isoformat(),
        "churn_probability": round(prob, 4),
        "risk_tier": tier,
        "tenure_segment": tenure_segment,
        "is_high_value": is_high_value,
        "monthly_spend": row["monthly_spend"],
        "recommended_action": action,
        "rule_fired": rule_fired,
        "true_churn_label": int(y_test.iloc[idx]),
    })

audit_df = pd.DataFrame(audit_records)
audit_df.to_csv("../data/audit_log.csv", index=False)

print(f"Processed {len(audit_df)} customers")
print(f"Customers with an action taken: {(audit_df['recommended_action'] != 'No action needed').sum()}")
print("Audit log saved to data/audit_log.csv")
audit_df.head()

Processed 1409 customers
Customers with an action taken: 586
Audit log saved to data/audit_log.csv


,customer_id,timestamp,churn_probability,risk_tier,tenure_segment,is_high_value,monthly_spend,recommended_action,rule_fired,true_churn_label
0,0,2026-09-01T15:35:16.921601,0.0418,Low,Loyal,True,114.05,No action needed,RULE_LOW_NO_ACTION,0
1,1,2026-09-01T15:35:16.921601,0.0291,Low,Established,True,100.15,No action needed,RULE_LOW_NO_ACTION,0
2,2,2026-09-01T15:35:16.921601,0.0615,Low,Loyal,False,78.35,No action needed,RULE_LOW_NO_ACTION,0
3,3,2026-09-01T15:35:16.921601,0.1433,Medium,Established,False,78.20,Proactive support outreach,RULE_MEDIUM_LOW_VALUE_OUTREACH,0
4,4,2026-09-01T15:35:16.921601,0.0670,Low,Loyal,False,82.65,No action needed,RULE_LOW_NO_ACTION,0


In [12]:
# Merge in the actual monthly_spend and true churn outcome for impact calculation
acted_on = audit_df[audit_df["recommended_action"] != "No action needed"].copy()

# Only true churners among those we acted on represent real "at-risk MRR" we could save
true_churners_acted_on = acted_on[acted_on["true_churn_label"] == 1]

mrr_at_risk_and_flagged = true_churners_acted_on["monthly_spend"].sum()
total_mrr_at_risk = audit_df[audit_df["true_churn_label"] == 1]["monthly_spend"].sum()

print(f"Customers flagged for action: {len(acted_on)}")
print(f"Of those, true churners (real at-risk MRR): {len(true_churners_acted_on)}")
print(f"Total MRR at risk (all true churners in test set): ${total_mrr_at_risk:,.2f}")
print(f"MRR at risk that we correctly flagged: ${mrr_at_risk_and_flagged:,.2f}")
print(f"Coverage: {mrr_at_risk_and_flagged/total_mrr_at_risk*100:.1f}% of at-risk MRR identified")

print("\n--- Estimated MRR SAVED at different retention success rates ---")
for success_rate in [0.20, 0.35, 0.50]:
    mrr_saved = mrr_at_risk_and_flagged * success_rate
    print(f"At {int(success_rate*100)}% retention success: ${mrr_saved:,.2f}/month saved")

Customers flagged for action: 586
Of those, true churners (real at-risk MRR): 367
Total MRR at risk (all true churners in test set): $27,214.90
MRR at risk that we correctly flagged: $26,712.20
Coverage: 98.2% of at-risk MRR identified

--- Estimated MRR SAVED at different retention success rates ---
At 20% retention success: $5,342.44/month saved
At 35% retention success: $9,349.27/month saved
At 50% retention success: $13,356.10/month saved


In [5]:
def risk_tier(prob):
    if prob < 0.10:
        return "Low"
    elif prob < 0.50:
        return "Medium"
    else:
        return "High"

# Apply to test set and check distribution
risk_tiers = [risk_tier(p) for p in churn_probs]
import pandas as pd
print(pd.Series(risk_tiers).value_counts())

Low       823
High      391
Medium    195
Name: count, dtype: int64


In [6]:
def recommend_action(risk_tier, is_high_value, tenure_segment):
    if risk_tier == "Low":
        return "No action needed"
    
    if risk_tier == "High":
        if is_high_value == 1 and tenure_segment == "Loyal":
            return "Proactive support outreach"
        elif is_high_value == 1:
            return "Personalized discount"
        else:
            return "Plan downgrade offer"
    
    if risk_tier == "Medium":
        if is_high_value == 1:
            return "Loyalty reward"
        else:
            return "Proactive support outreach"

# Quick test with a few manual examples
print(recommend_action("High", 1, "Loyal"))       # expect: Proactive support outreach
print(recommend_action("High", 1, "New"))         # expect: Personalized discount
print(recommend_action("High", 0, "Established")) # expect: Plan downgrade offer
print(recommend_action("Medium", 1, "Loyal"))      # expect: Loyalty reward
print(recommend_action("Medium", 0, "New"))        # expect: Proactive support outreach
print(recommend_action("Low", 0, "New"))           # expect: No action needed

Proactive support outreach
Personalized discount
Plan downgrade offer
Loyalty reward
Proactive support outreach
No action needed


In [7]:
def churn_retention_agent(customer_row, model):
    # Get churn probability from the model
    prob = model.predict_proba(customer_row.values.reshape(1, -1))[0, 1]
    
    # Determine risk tier
    tier = risk_tier(prob)
    
    # Get segment info from the row
    is_high_value = customer_row["is_high_value_customer"]
    
    # NOTE: tenure_segment isn't in X_train (we dropped it in Step 4.1)
    # so we need to derive it fresh from customer_tenure_months
    tenure_months = customer_row["customer_tenure_months"]
    if tenure_months <= 6:
        t_segment = "New"
    elif tenure_months <= 24:
        t_segment = "Established"
    else:
        t_segment = "Loyal"
    
    # Get the recommended action
    action = recommend_action(tier, is_high_value, t_segment)
    
    return {
        "churn_probability": round(prob, 4),
        "risk_tier": tier,
        "tenure_segment": t_segment,
        "is_high_value": bool(is_high_value),
        "recommended_action": action
    }

# Test on one customer from the test set
sample_customer = X_test.iloc[0]
result = churn_retention_agent(sample_customer, best_xgb)
print(result)

{'churn_probability': np.float32(0.0418), 'risk_tier': 'Low', 'tenure_segment': 'Loyal', 'is_high_value': True, 'recommended_action': 'No action needed'}


In [8]:
def generate_explanation(risk_tier, is_high_value, tenure_segment, prob):
    if risk_tier == "Low":
        return f"Low churn risk ({prob:.1%}). Customer appears stable — no intervention needed."
    
    if risk_tier == "High" and is_high_value and tenure_segment == "Loyal":
        return f"High churn risk ({prob:.1%}) despite being a loyal, high-value customer — likely an unresolved issue. Recommend direct outreach rather than a discount, since retention here isn't a pricing problem."
    
    if risk_tier == "High" and is_high_value:
        return f"High churn risk ({prob:.1%}) in a high-value but not-yet-loyal customer. A personalized discount may reinforce their decision to stay before habits form."
    
    if risk_tier == "High":
        return f"High churn risk ({prob:.1%}) in a lower-value customer. A cheaper plan option may retain them at lower cost than a discount."
    
    if risk_tier == "Medium" and is_high_value:
        return f"Moderate churn risk ({prob:.1%}) in a high-value customer. A loyalty reward can reinforce the relationship proactively."
    
    if risk_tier == "Medium":
        return f"Moderate churn risk ({prob:.1%}). A light-touch check-in may catch emerging issues early."

# Test it with the same sample customer
explanation = generate_explanation(result["risk_tier"], result["is_high_value"], 
                                     result["tenure_segment"], result["churn_probability"])
print(explanation)

Low churn risk (4.2%). Customer appears stable — no intervention needed.


In [9]:
def churn_retention_agent(customer_row, model):
    prob = model.predict_proba(customer_row.values.reshape(1, -1))[0, 1]
    tier = risk_tier(prob)
    is_high_value = customer_row["is_high_value_customer"]
    
    tenure_months = customer_row["customer_tenure_months"]
    if tenure_months <= 6:
        t_segment = "New"
    elif tenure_months <= 24:
        t_segment = "Established"
    else:
        t_segment = "Loyal"
    
    action = recommend_action(tier, is_high_value, t_segment)
    explanation = generate_explanation(tier, bool(is_high_value), t_segment, prob)
    
    return {
        "churn_probability": round(float(prob), 4),
        "risk_tier": tier,
        "tenure_segment": t_segment,
        "is_high_value": bool(is_high_value),
        "recommended_action": action,
        "explanation": explanation
    }

In [10]:
def churn_retention_agent(customer_row, model):
    prob = model.predict_proba(customer_row.values.reshape(1, -1))[0, 1]
    tier = risk_tier(prob)
    is_high_value = customer_row["is_high_value_customer"]
    
    tenure_months = customer_row["customer_tenure_months"]
    if tenure_months <= 6:
        t_segment = "New"
    elif tenure_months <= 24:
        t_segment = "Established"
    else:
        t_segment = "Loyal"
    
    action = recommend_action(tier, is_high_value, t_segment)
    explanation = generate_explanation(tier, bool(is_high_value), t_segment, prob)
    
    return {
        "churn_probability": round(float(prob), 4),
        "risk_tier": tier,
        "tenure_segment": t_segment,
        "is_high_value": bool(is_high_value),
        "recommended_action": action,
        "explanation": explanation
    }

In [11]:
# Find a genuinely high-risk customer in the test set to test with
high_risk_idx = np.argmax(churn_probs)
sample_high_risk = X_test.iloc[high_risk_idx]

result = churn_retention_agent(sample_high_risk, best_xgb)
for key, value in result.items():
    print(f"{key}: {value}")

NameError: name 'np' is not defined

In [ ]:
# Run the agent on the first 20 customers in the test set
results = []
for i in range(20):
    row = X_test.iloc[i]
    res = churn_retention_agent(row, best_xgb)
    res["customer_index"] = i
    results.append(res)

results_df = pd.DataFrame(results)
results_df = results_df[["customer_index", "churn_probability", "risk_tier", 
                          "tenure_segment", "is_high_value", "recommended_action"]]
print(results_df)

# Quick summary: how many customers per action?
print("\nAction distribution:")
print(results_df["recommended_action"].value_counts())

    customer_index  churn_probability risk_tier tenure_segment  is_high_value  \
0                0             0.0418       Low          Loyal           True   
1                1             0.0291       Low    Established           True   
2                2             0.0615       Low          Loyal          False   
3                3             0.1433    Medium    Established          False   
4                4             0.0670       Low          Loyal          False   
5                5             0.2903    Medium    Established           True   
6                6             0.0156       Low    Established           True   
7                7             0.5261      High    Established          False   
8                8             0.0148       Low          Loyal          False   
9                9             0.7251      High    Established          False   
10              10             0.0091       Low          Loyal           True   
11              11          